# Champollion PBMC tutorial

This tutorial demonstrates how to use Champollion on a PBMC multiome RNA/ATAC dataset. Because this benchmark dataset is fully paired, every cell has both modalities. To mimic the setting Champollion is designed for, we use part of the cells as the paired bridge and treat the remaining cells as pseudo-unpaired RNA and ATAC profiles.

The dataset used here is already preprocessed. For guidance on preparing a new dataset, see the Data Inputs section of the documentation.

In a typical application, the bridge would come from a paired assay such as multiome, while the unpaired cells could come from separate scRNA-seq and scATAC-seq experiments. Champollion learns the cross-modality cost on the bridge, then uses it to integrate the unpaired cells, transfer annotations, and build joint visualizations.

In [ ]:
import os

# Keep local notebook execution from writing caches into non-writable home folders.
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba")
os.environ.setdefault("KEOPS_CACHE_FOLDER", "/tmp/keops2.3")

In [ ]:
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import muon as mu
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from matplotlib.colors import LogNorm, PowerNorm
from sklearn.metrics import accuracy_score, confusion_matrix

from champollion import Champollion

## Configuration

We first define the modality names, representations, and a small set of model parameters. The representations used here are low-dimensional RNA/ATAC embeddings stored in `.obsm`; the prior representation provides a common space used to compute a prior cost, it was obtained by computing gene activities for ATAC profiles with `Signac`.

In [ ]:
DATA_PATH = Path("../semi-paired/data/pbmc_all/incomplete.h5mu")

RNA_MOD = "rna"
ATAC_MOD = "atac"
CELLTYPE_KEY = "celltype"
PAIRED_MASK_KEY = "paired_mask"

MAIN_REP = "obsm/X_pca"
PRIOR_REP = "obsm/prior_data"

# Debug mode keeps the notebook executable on a laptop. Set DEBUG = False on the cluster.
DEBUG = False
DEBUG_TRAIN_CELLS = 120
DEBUG_TEST_CELLS = 180

MODEL_KWARGS = dict(
    epsilon=1.0,
    gamma=0.01,
    lambda_prior=20.0,
    use_keops=False,
    device="auto",
    random_state=0,
    max_iter=3 if DEBUG else 300_000,
    learning_rate=1e-3,
    sinkhorn_tol=1e-3,
    log_every=1 if DEBUG else 40,
    verbose=True,
)

MAX_MATERIALIZE_ENTRIES = 50_000 * 50_000

## Load and format the data

The file contains a single fully paired PBMC multiome object. The `paired_mask` column is used here only to construct a tutorial split: cells marked as paired form the bridge, and the remaining cells are copied into two modality-specific `AnnData` objects that we deliberately treat as unpaired.

In [ ]:
mdata = mu.read_h5mu(DATA_PATH)
print(mdata)
print("modalities:", list(mdata.mod.keys()))

In [ ]:
paired_mask = np.asarray(mdata.obs[PAIRED_MASK_KEY].to_numpy(), dtype=bool)
celltypes = mdata.obs[CELLTYPE_KEY].astype(str).to_numpy()

print("paired training cells:", int(paired_mask.sum()))
print("held-out cells:", int((~paired_mask).sum()))
pd.crosstab(mdata.obs[CELLTYPE_KEY], mdata.obs[PAIRED_MASK_KEY], colnames=["paired"])

In [ ]:
def stratified_indices(mask, labels, n=None, random_state=0):
    """Return indices within mask, optionally stratified down to n cells."""
    idx = np.flatnonzero(mask)
    if n is None or n >= len(idx):
        return idx
    rng = np.random.default_rng(random_state)
    selected = []
    present_labels = [
        label for label in pd.unique(labels[idx]) if np.any(labels[idx] == label)
    ]
    per_group = max(1, int(np.ceil(n / len(present_labels))))
    for label in present_labels:
        label_idx = idx[labels[idx] == label]
        take = min(per_group, len(label_idx))
        selected.extend(rng.choice(label_idx, size=take, replace=False))
    selected = np.asarray(selected, dtype=int)
    if len(selected) > n:
        selected = rng.choice(selected, size=n, replace=False)
    return np.sort(selected)


train_idx = stratified_indices(
    paired_mask,
    labels=celltypes,
    n=DEBUG_TRAIN_CELLS if DEBUG else None,
    random_state=0,
)
test_idx = stratified_indices(
    ~paired_mask,
    labels=celltypes,
    n=DEBUG_TEST_CELLS if DEBUG else None,
    random_state=1,
)

train_mdata = mu.MuData(
    {
        RNA_MOD: mdata.mod[RNA_MOD][train_idx].copy(),
        ATAC_MOD: mdata.mod[ATAC_MOD][train_idx].copy(),
    }
)

adata_rna = mdata.mod[RNA_MOD][test_idx].copy()
adata_atac = mdata.mod[ATAC_MOD][test_idx].copy()

print(train_mdata)
print("held-out RNA:", adata_rna.shape)
print("held-out ATAC:", adata_atac.shape)

## Fit Champollion on the paired bridge

Champollion learns the sparse cross-modality matrix `A` from the bridge cells. The bridge contains matched RNA and ATAC profiles for the same cells, so it provides the supervision needed to learn how the two representation spaces should be compared.

The verbose optimization log from the server run is shortened in this rendered notebook to keep the tutorial readable.

In [ ]:
model = Champollion(**MODEL_KWARGS)
model.fit(
    train_mdata,
    modality_1=RNA_MOD,
    modality_2=ATAC_MOD,
    x_1_rep=MAIN_REP,
    x_2_rep=MAIN_REP,
    y_prior_1_rep=PRIOR_REP,
    y_prior_2_rep=PRIOR_REP,
)

print("A shape:", tuple(model.A_.shape))
print("A device:", model.A_.device)
pd.DataFrame(
    {key: pd.Series(value) for key, value in model.training_history_.items()}
).tail()

## Integrate pseudo-unpaired cells

We now apply the learned cost to the RNA and ATAC cells held out from the bridge. Although these cells are paired in the original multiome data, Champollion only sees them as two separate `AnnData` objects keyed by modality name.

In [ ]:
transport = model.transport(
    {RNA_MOD: adata_rna, ATAC_MOD: adata_atac},
    x_reps={RNA_MOD: MAIN_REP, ATAC_MOD: MAIN_REP},
    y_prior_reps={RNA_MOD: PRIOR_REP, ATAC_MOD: PRIOR_REP},
    store_cost=True,
    store_plan=True,
    max_iter_sink=30 if DEBUG else 1000,
)

plan = (
    transport.materialize_plan(max_entries=MAX_MATERIALIZE_ENTRIES)
    .detach()
    .cpu()
    .numpy()
)
print("plan shape:", plan.shape)
print("plan mass:", plan.sum())
transport.plan_diagnostics

## Inspect the transport plan

The transport plan summarizes how mass is assigned between RNA and ATAC cells. Reordering cells by known labels is a useful diagnostic: a good integration should concentrate mass around biologically matching populations.

In [ ]:
def plot_ordered_transport_plan(
    heat_mtx,
    celltypes,
    ctypes_ordered,
    celltypes_2=None,
    ctypes_ordered_2=None,
    normalize_rows=False,
    cmap="magma",
    figsize=(7, 7),
    show_boundaries=True,
    show_celltype_ticks=True,
    tick_fontsize=10,
    linecolor="white",
    linewidth=1.0,
    title=None,
    norm_method="log",
    power=0.4,
    vmin=1e-8,
    vmax=None,
    xlabel="ATAC cells",
    ylabel="RNA cells",
):
    heat_mtx = np.asarray(heat_mtx)
    celltypes = np.asarray(celltypes)
    if celltypes_2 is None:
        celltypes_2 = celltypes
        ctypes_ordered_2 = ctypes_ordered
    else:
        celltypes_2 = np.asarray(celltypes_2)
        if ctypes_ordered_2 is None:
            raise ValueError(
                "If celltypes_2 is provided, ctypes_ordered_2 must also be provided."
            )
    if heat_mtx.shape[0] != len(celltypes):
        raise ValueError("heat_mtx.shape[0] must match len(celltypes).")
    if heat_mtx.shape[1] != len(celltypes_2):
        raise ValueError("heat_mtx.shape[1] must match len(celltypes_2).")

    row_groups = [np.where(celltypes == ct)[0] for ct in ctypes_ordered]
    col_groups = [np.where(celltypes_2 == ct)[0] for ct in ctypes_ordered_2]
    row_order = np.concatenate(row_groups) if row_groups else np.array([], dtype=int)
    col_order = np.concatenate(col_groups) if col_groups else np.array([], dtype=int)
    ordered_mtx = heat_mtx[np.ix_(row_order, col_order)].astype(float, copy=True)

    if normalize_rows:
        row_sums = np.nansum(ordered_mtx, axis=1, keepdims=True)
        ordered_mtx = np.divide(
            ordered_mtx,
            row_sums,
            out=np.full_like(ordered_mtx, np.nan, dtype=float),
            where=row_sums > 0,
        )

    fig, ax = plt.subplots(figsize=figsize)
    if norm_method == "log":
        plot_mtx = ordered_mtx.copy()
        plot_mtx[plot_mtx <= 0] = vmin
        im = ax.imshow(
            plot_mtx,
            cmap=cmap,
            norm=LogNorm(vmin=vmin, vmax=vmax or np.nanmax(plot_mtx)),
        )
    else:
        im = ax.imshow(ordered_mtx, cmap=cmap, norm=PowerNorm(gamma=power))

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title is None:
        title = "Transport plan ordered by cell type"
        if normalize_rows:
            title += " (row-normalized)"
    ax.set_title(title)

    if show_boundaries:
        for b in np.cumsum([len(g) for g in row_groups])[:-1]:
            ax.axhline(b - 0.5, color=linecolor, linewidth=linewidth)
        for b in np.cumsum([len(g) for g in col_groups])[:-1]:
            ax.axvline(b - 0.5, color=linecolor, linewidth=linewidth)

    if show_celltype_ticks:
        row_sizes = [len(g) for g in row_groups]
        col_sizes = [len(g) for g in col_groups]
        row_starts = np.cumsum([0] + row_sizes[:-1])
        col_starts = np.cumsum([0] + col_sizes[:-1])
        ax.set_yticks(row_starts + np.asarray(row_sizes) / 2 - 0.5)
        ax.set_yticklabels(ctypes_ordered, fontsize=tick_fontsize)
        ax.set_xticks(col_starts + np.asarray(col_sizes) / 2 - 0.5)
        ax.set_xticklabels(
            ctypes_ordered_2, rotation=45, ha="right", fontsize=tick_fontsize
        )
    else:
        ax.set_xticks([])
        ax.set_yticks([])

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(
        "Transport mass" if not normalize_rows else "Row-normalized transport mass"
    )
    plt.tight_layout()
    return ordered_mtx, row_order, col_order, fig, ax

In [ ]:
rna_labels = adata_rna.obs[CELLTYPE_KEY].astype(str).to_numpy()
atac_labels = adata_atac.obs[CELLTYPE_KEY].astype(str).to_numpy()
celltype_order = sorted(pd.unique(np.concatenate([rna_labels, atac_labels])))

_, _, _, fig, ax = plot_ordered_transport_plan(
    heat_mtx=plan,
    celltypes=rna_labels,
    ctypes_ordered=celltype_order,
    celltypes_2=atac_labels,
    ctypes_ordered_2=celltype_order,
    normalize_rows=False,
    cmap="magma",
    figsize=(7, 7),
    vmin=1e-16,
    norm_method="log",
    xlabel="ATAC cells",
    ylabel="RNA cells",
)
plt.show()

## Transfer cell type annotations from RNA to ATAC

A common use case is to transfer annotations from a well-annotated modality to another modality. Here we transfer RNA cell type labels to ATAC cells through the transport plan and compare the predictions with the known labels from the paired dataset.

In [ ]:
transfer = transport.transfer_obs(
    CELLTYPE_KEY,
    source=RNA_MOD,
    kind="categorical",
    inplace=True,
    target_key="celltype_champollion",
    return_probabilities=True,
)

predicted = transfer["prediction"].astype(str)
true = adata_atac.obs[CELLTYPE_KEY].astype(str)
print("cell type transfer accuracy:", accuracy_score(true, predicted))
transfer["probabilities"].head()

In [ ]:
labels = [
    label
    for label in celltype_order
    if (true.eq(label).any() or predicted.eq(label).any())
]
cm = confusion_matrix(true, predicted, labels=labels)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels,
    cbar=False,
    ax=ax,
)
ax.set_xlabel("Predicted ATAC label")
ax.set_ylabel("True ATAC label")
ax.set_title("RNA-to-ATAC cell type transfer")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## Barycentric projection into RNA PCA space

The same transport plan can project continuous representations across modalities. Here each ATAC cell is represented as a barycenter of RNA cells in RNA PCA space, which enables a joint visualization of original RNA cells and projected ATAC cells.

In [ ]:
projected_atac_in_rna_pca = transport.project(rep=MAIN_REP, source=RNA_MOD)

joint = ad.AnnData(X=np.vstack([adata_rna.obsm["X_pca"], projected_atac_in_rna_pca]))
joint.obs["modality"] = pd.Categorical(
    ["RNA"] * adata_rna.n_obs + ["ATAC projected"] * adata_atac.n_obs
)
joint.obs[CELLTYPE_KEY] = pd.Categorical(np.concatenate([rna_labels, atac_labels]))

sc.pp.neighbors(joint, n_neighbors=15, use_rep="X")
sc.tl.umap(joint, min_dist=0.2, random_state=0)

sc.pl.umap(joint, color=["modality", CELLTYPE_KEY], alpha=0.65, wspace=0.35)